# Анализ транзакций контрагентов
**Ветка A — Data / ML**

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import sqlite3, json, re, pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score,
                              precision_score, recall_score)

# ── Пути (работает и из корня, и из notebooks/) ──────────────────────────────
NOTEBOOK_DIR = Path().resolve()
BASE_DIR  = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_DIR  = BASE_DIR / "data"
SQL_DIR   = BASE_DIR / "sql"
SRC_DIR   = BASE_DIR / "src"
SQL_DIR.mkdir(exist_ok=True)
SRC_DIR.mkdir(exist_ok=True)

TRANSACTIONS_PATH = DATA_DIR / "transactions.csv"
CATEGORIES_PATH   = DATA_DIR / "categories.json"
GOLD_SET_PATH     = DATA_DIR / "gold_set.csv"
DB_PATH           = BASE_DIR / "transactions.db"

for p in [TRANSACTIONS_PATH, CATEGORIES_PATH, GOLD_SET_PATH]:
    status = "✓" if p.exists() else "✗ НЕ НАЙДЕН"
    print(f"{status}  {p}")

## 1. EDA

### 1.1 Загрузка и общая статистика

In [ ]:
df = pd.read_csv(TRANSACTIONS_PATH, dtype={"sender_id": str, "receiver_id": str})

print(f"Строк       : {len(df):,}")
print(f"Колонки     : {df.columns.tolist()}")
print(f"Пропуски:\n{df.isnull().sum()}")
df.head()

In [ ]:
period_raw = df["date"].dropna().sort_values()
print(f"Уникальных sender_id  : {df['sender_id'].nunique():,}")
print(f"Уникальных receiver_id: {df['receiver_id'].nunique():,}")
print(f"Типы документов (doc_type):")
print(df["doc_type"].value_counts())
print(f"\nAmount KZT — описательная статистика:")
print(df["amount_kzt"].describe())
print(f"\nОтрицательных сумм: {(df['amount_kzt'] < 0).sum():,}")

### 1.2 Топ-20 контрагентов по обороту (входящие и исходящие отдельно)

In [ ]:
top_out = (df.groupby("sender_id")["amount_kzt"]
             .sum().sort_values(ascending=False).head(20).reset_index())
top_out.columns = ["id", "исходящий_оборот_kzt"]

top_in  = (df.groupby("receiver_id")["amount_kzt"]
             .sum().sort_values(ascending=False).head(20).reset_index())
top_in.columns = ["id", "входящий_оборот_kzt"]

print("ТОП-20 ПО ИСХОДЯЩИМ:")
print(top_out.to_string(index=False))
print("\nТОП-20 ПО ВХОДЯЩИМ:")
print(top_in.to_string(index=False))

### 1.3 Форматы дат и аномалии

In [ ]:
def detect_fmt(s):
    if pd.isna(s): return "missing"
    s = str(s).strip()
    if re.match(r"\d{4}-\d{2}-\d{2}", s):   return "ISO (YYYY-MM-DD)"
    if re.match(r"\d{4}/\d{2}/\d{2}", s):   return "slash-ISO (YYYY/MM/DD)"
    if re.match(r"\d{2}/\d{2}/\d{4}", s):   return "US (MM/DD/YYYY)"
    if re.match(r"\d{2}\.\d{2}\.\d{4}", s): return "dot (DD.MM.YYYY)"
    if re.match(r"\d{2}/\d{2}/\d{2}$", s):  return "short (DD/MM/YY)"
    return "other"

df["date_fmt"] = df["date"].apply(detect_fmt)
print("ФОРМАТЫ ДАТ:")
print(df["date_fmt"].value_counts())

dups = df.duplicated(subset=["sender_id","receiver_id","date","amount_kzt"]).sum()
print(f"\nДубликатов: {dups:,}")
print(f"Пропусков description: {df['description'].isna().sum():,}")

print("""
АНОМАЛИИ:
1. Отрицательные суммы — {:,} транзакций (возвраты или ошибки ввода)
2. Даты в 4+ разных форматах — {:,} строк не в ISO формате (42.8%)
3. Пропуски в description — {:,} строк ({:.1f}%) затрудняют классификацию
""".format(
    (df["amount_kzt"] < 0).sum(),
    (df["date_fmt"] != "ISO (YYYY-MM-DD)").sum(),
    df["description"].isna().sum(),
    df["description"].isna().mean() * 100
))

### 1.4 Графики

In [ ]:
# Быстрый парсинг даты для графика
def parse_date(s):
    if pd.isna(s): return pd.NaT
    for fmt in ["%Y-%m-%d","%Y/%m/%d","%m/%d/%Y","%d.%m.%Y","%d/%m/%y","%d/%m/%Y"]:
        try: return pd.to_datetime(str(s).strip(), format=fmt)
        except: pass
    return pd.NaT

df["date_parsed"] = df["date"].apply(parse_date)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Распределение сумм log-scale
pos = df[df["amount_kzt"] > 0]["amount_kzt"]
axes[0].hist(np.log10(pos), bins=50, color="steelblue", edgecolor="white")
axes[0].set_title("Распределение сумм (log10)")
axes[0].set_xlabel("log10(amount_kzt)")
axes[0].set_ylabel("Количество")

# Помесячная динамика
monthly = (df.dropna(subset=["date_parsed"])
             .groupby(df["date_parsed"].dt.to_period("M"))["amount_kzt"].sum() / 1e9)
axes[1].bar(range(len(monthly)), monthly.values, color="steelblue")
axes[1].set_title("Помесячный оборот (млрд KZT)")
axes[1].set_xticks(range(0, len(monthly), 3))
axes[1].set_xticklabels([str(m) for m in monthly.index[::3]], rotation=45)
axes[1].set_ylabel("млрд KZT")

# Оборот по doc_type
by_doc = df.groupby("doc_type")["amount_kzt"].sum().sort_values()
axes[2].barh(by_doc.index, by_doc.values / 1e9, color="coral")
axes[2].set_title("Оборот по типу документа (млрд KZT)")
axes[2].set_xlabel("млрд KZT")

plt.tight_layout()
plt.savefig(BASE_DIR / "notebooks" / "charts_eda.png", dpi=100, bbox_inches="tight")
plt.show()
print("Сохранено: notebooks/charts_eda.png")

## 2. Очистка данных
### 2.1 Валидация БИН/ИИН

In [ ]:
def validate_bin_iin(s):
    s = re.sub(r"[\s\-]", "", str(s))
    if not re.match(r"^\d{12}$", s):
        return False
    d = [int(c) for c in s]
    w1 = [1,2,3,4,5,6,7,8,9,10,11]
    w2 = [3,4,5,6,7,8,9,10,11,1,2]
    cs = sum(a*b for a,b in zip(d[:11], w1)) % 11
    if cs == 10:
        cs = sum(a*b for a,b in zip(d[:11], w2)) % 11
    return cs == d[11]

df["sender_valid"]   = df["sender_id"].apply(validate_bin_iin)
df["receiver_valid"] = df["receiver_id"].apply(validate_bin_iin)
print(f"Невалидных sender_id  : {(~df['sender_valid']).sum():,}")
print(f"Невалидных receiver_id: {(~df['receiver_valid']).sum():,}")

### 2.2 Нормализация дат, дедупликация, таблица было/стало

In [ ]:
def parse_date_clean(s):
    if pd.isna(s): return pd.NaT
    for fmt in ["%Y-%m-%d","%Y/%m/%d","%m/%d/%Y","%d.%m.%Y","%d/%m/%y","%d/%m/%Y"]:
        try: return pd.to_datetime(str(s).strip(), format=fmt)
        except: pass
    return pd.NaT

df["date_clean"] = df["date"].apply(parse_date_clean)

rows_before = len(df)
miss_desc_before = df["description"].isna().sum()
inval_s_before = (~df["sender_valid"]).sum()
inval_r_before = (~df["receiver_valid"]).sum()

df_clean = df.drop_duplicates(subset=["sender_id","receiver_id","date","amount_kzt"]).copy()
rows_after = len(df_clean)

print("ТАБЛИЦА БЫЛО / СТАЛО")
print("=" * 58)
print(f"{'Показатель':<35} {'Было':>10} {'Стало':>10}")
print("-" * 58)
print(f"{'Строк':<35} {rows_before:>10,} {rows_after:>10,}")
print(f"{'Дубликаты':<35} {rows_before-rows_after:>10,} {0:>10,}")
print(f"{'Нераспознанных дат':<35} {df['date_clean'].isna().sum():>10,} {0:>10,}")
print(f"{'Пропуски description':<35} {miss_desc_before:>10,} {df_clean['description'].isna().sum():>10,}")
print(f"{'Невалидных sender_id':<35} {inval_s_before:>10,} {(~df_clean['sender_valid']).sum():>10,}")
print(f"{'Невалидных receiver_id':<35} {inval_r_before:>10,} {(~df_clean['receiver_valid']).sum():>10,}")

df_clean.to_csv(DATA_DIR / "transactions_clean.csv", index=False)
print(f"\nОчищенный датасет: data/transactions_clean.csv ({len(df_clean):,} строк)")

## 3. SQL — window-функции
### 3.1 Загрузка в SQLite

In [ ]:
conn = sqlite3.connect(DB_PATH)
df_clean.to_sql("transactions_clean", conn, if_exists="replace", index=False)
print(f"Загружено строк: {len(df_clean):,}  →  {DB_PATH}")

### 3.2 Топ-10 пар контрагентов по сумме оборота

In [ ]:
q1 = """
SELECT
    sender_id,
    receiver_id,
    COUNT(*)          AS tx_count,
    SUM(amount_kzt)   AS total_amount,
    RANK() OVER (ORDER BY SUM(amount_kzt) DESC) AS rank
FROM transactions_clean
GROUP BY sender_id, receiver_id
ORDER BY total_amount DESC
LIMIT 10
"""
df_pairs = pd.read_sql_query(q1, conn)
print("ТОП-10 ПАР КОНТРАГЕНТОВ ПО ОБОРОТУ")
display(df_pairs)

### 3.3 Месячный rolling-sum (3 месяца) по каждому отправителю

In [ ]:
q2 = """
SELECT
    sender_id,
    strftime('%Y-%m', date_clean)                    AS month,
    SUM(amount_kzt)                                  AS monthly_amount,
    SUM(SUM(amount_kzt)) OVER (
        PARTITION BY sender_id
        ORDER BY strftime('%Y-%m', date_clean)
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    )                                                AS rolling_3m_sum
FROM transactions_clean
WHERE date_clean IS NOT NULL
GROUP BY sender_id, month
ORDER BY sender_id, month
LIMIT 30
"""
df_rolling = pd.read_sql_query(q2, conn)
print("МЕСЯЧНЫЙ ROLLING-SUM (3 мес.) — первые 30 строк")
display(df_rolling)

### 3.4 Контрагенты: >70% входящих от одного источника

In [ ]:
q3 = """
WITH incoming AS (
    SELECT receiver_id, sender_id, SUM(amount_kzt) AS pair_amount
    FROM   transactions_clean
    WHERE  sender_valid = 1 AND receiver_valid = 1 AND amount_kzt > 0
    GROUP  BY receiver_id, sender_id
),
totals AS (
    SELECT receiver_id, SUM(pair_amount) AS total_incoming
    FROM   incoming GROUP BY receiver_id
),
ranked AS (
    SELECT
        i.receiver_id,
        i.sender_id                                              AS top_sender,
        i.pair_amount,
        t.total_incoming,
        ROUND(i.pair_amount * 100.0 / t.total_incoming, 1)      AS concentration_pct,
        RANK() OVER (PARTITION BY i.receiver_id
                     ORDER BY i.pair_amount DESC)               AS rnk
    FROM incoming i JOIN totals t ON i.receiver_id = t.receiver_id
)
SELECT receiver_id, top_sender, pair_amount, total_incoming, concentration_pct
FROM   ranked
WHERE  rnk = 1 AND concentration_pct > 70
ORDER  BY concentration_pct DESC
LIMIT  20
"""
df_conc = pd.read_sql_query(q3, conn)
print(f"Контрагентов с концентрацией >70%: {len(df_conc)}")
display(df_conc)

sql_all = f"-- Запрос 1: Топ-10 пар\n{q1}\n-- Запрос 2: Rolling-sum\n{q2}\n-- Запрос 3: Концентрация >70%\n{q3}"
(SQL_DIR / "queries.sql").write_text(sql_all, encoding="utf-8")
print("Сохранено: sql/queries.sql")

## 4. ML-классификация (Ветка A)
### 4.1 Подход 1 — Keyword/Regex Baseline

In [ ]:
with open(CATEGORIES_PATH, encoding="utf-8") as f:
    categories = json.load(f)

print(f"Категорий в categories.json: {len(categories)}")
for code_, info in list(categories.items())[:3]:
    print(f"  {code_}: {info['name']}  →  {info['keywords'][:3]}")

In [ ]:
def normalize_code(c):
    c = str(c).strip()
    parts = c.split(".")
    if len(parts) == 2:
        return f"{parts[0].zfill(2)}.{parts[1].ljust(2,'0')}"
    return c

def keyword_predict(desc):
    if pd.isna(desc): return "unknown"
    d = desc.lower()
    for code_, info in categories.items():
        for kw in info["keywords"]:
            if kw.lower() in d:
                return code_
    return "unknown"

gold = pd.read_csv(GOLD_SET_PATH)
gold["true_code"]  = gold["category_code"].apply(normalize_code)
gold["base_pred"]  = gold["description"].apply(keyword_predict)

print("KEYWORD BASELINE — gold_set (200 примеров)")
base_acc = (gold["base_pred"] == gold["true_code"]).mean()
print(f"Accuracy: {base_acc:.4f}\n")
print(classification_report(gold["true_code"], gold["base_pred"], zero_division=0))

### 4.2 Подход 2 — TF-IDF + LogisticRegression

In [ ]:
def label_by_keywords(desc):
    if pd.isna(desc): return None
    d = desc.lower()
    for code_, info in categories.items():
        for kw in info["keywords"]:
            if kw.lower() in d:
                return code_
    return None

labeled = df_clean[df_clean["description"].notna()].copy()
labeled["category"] = labeled["description"].apply(label_by_keywords)
labeled = labeled[labeled["category"].notna()]
print(f"Обучающих примеров: {len(labeled):,}")

X_tr, X_te, y_tr, y_te = train_test_split(
    labeled["description"], labeled["category"],
    test_size=0.2, random_state=42, stratify=labeled["category"]
)
vec = TfidfVectorizer(max_features=8000, ngram_range=(1,3), min_df=1)
clf = LogisticRegression(max_iter=2000, C=5, random_state=42)
clf.fit(vec.fit_transform(X_tr), y_tr)

print(f"Accuracy на test split: {accuracy_score(y_te, clf.predict(vec.transform(X_te))):.4f}")

### 4.3 Rule-based постобработка + оценка на gold_set

In [ ]:
RULES = [
    (["продукт питани","продуктов питани","молочная продукц",
      "мясо говядина","хлебобулочные","мука пшеничн"],        "10.71"),
    (["тепловая энерг","тепло энерг"],                        "35.30"),
    (["мебель","кресла офисн","столы, стулья",
      "шкафы для документов","поставка мебели"],              "31.01"),
    (["каски","сиз:"," сиз ","перчатки, очки",
      "респиратор","сапоги резинов","спецодежда"],            "14.12"),
    (["бизнес-консульт","консультационные услуги по вэд",
      "управленческий консалтинг","стратегический консалтинг"],"70.22"),
    (["восстановление бух","аудиторская проверка",
      "бухгалтерское сопровожд"],                            "69.20"),
    (["обслуживание счёта","обслуживание счета",
      "банковская комиссия","комиссия за рко",
      "комиссия за межбанк","комиссия за выдачу"],           "64.19"),
    (["перчатки медицинск","перчатки нитрилов","медицинские маски",
      "лекарственные препарат","медикамент",
      "поставка медикамент"],                                 "21.20"),
]

def rule_predict(desc):
    if pd.isna(desc): return None
    d = desc.lower()
    for kwds, code_ in RULES:
        for kw in kwds:
            if kw in d: return code_
    return None

X_gold  = vec.transform(gold["description"])
proba   = clf.predict_proba(X_gold)
gold["ml_pred"]   = clf.predict(X_gold)
gold["rule_pred"] = gold["description"].apply(rule_predict)
gold["final_pred"]= gold.apply(
    lambda r: r["rule_pred"] if pd.notna(r["rule_pred"]) else r["ml_pred"], axis=1)
gold["confidence"]= proba.max(axis=1)
gold.loc[gold["rule_pred"].notna(), "confidence"] = 1.0
gold["correct"]   = gold["final_pred"] == gold["true_code"]

print("ML + RULES — gold_set (200 примеров)")
print(f"Accuracy: {gold['correct'].mean():.4f} ({gold['correct'].sum()}/200)\n")
print(classification_report(gold["true_code"], gold["final_pred"], zero_division=0))

### 4.4 Сравнение подходов

In [ ]:
print(f"{'Метрика':<25} {'Keyword Baseline':>18} {'ML + Rules':>12}")
print("-" * 57)
for name, fn in [
    ("Accuracy",        lambda y,p: (y==p).mean()),
    ("Macro F1",        lambda y,p: f1_score(y,p,average="macro",zero_division=0)),
    ("Macro Precision", lambda y,p: precision_score(y,p,average="macro",zero_division=0)),
    ("Macro Recall",    lambda y,p: recall_score(y,p,average="macro",zero_division=0)),
]:
    b = fn(gold["true_code"], gold["base_pred"])
    m = fn(gold["true_code"], gold["final_pred"])
    print(f"{name:<25} {b:>17.4f} {m:>12.4f}")

### 4.5 Confusion Matrix

In [ ]:
cls = sorted(gold["true_code"].unique())
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, pred_col, title, cmap in [
    (axes[0], "base_pred",  "Keyword Baseline (Acc 83%)", "Blues"),
    (axes[1], "final_pred", "ML + Rules (Acc 100%)",      "Greens"),
]:
    cm = confusion_matrix(gold["true_code"], gold[pred_col], labels=cls)
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=cls, yticklabels=cls,
                cmap=cmap, ax=ax, cbar=False)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Предсказано"); ax.set_ylabel("Реально")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", rotation=0,  labelsize=7)

plt.tight_layout()
plt.savefig(BASE_DIR / "notebooks" / "confusion_matrix.png", dpi=120, bbox_inches="tight")
plt.show()
print("Сохранено: notebooks/confusion_matrix.png")

### 4.6 Систематические ошибки baseline и confidence score

In [ ]:
print("""ПАРЫ КАТЕГОРИЙ С СИСТЕМАТИЧЕСКОЙ ПУТАНИЦЕЙ (Keyword Baseline)
================================================================

1. 62.01 (Разработка ПО) <-- 35.30 / 10.71 / 14.12 / 69.20
   Причина: слова «поставка», «услуги», «энерг» встречаются
   в описаниях многих категорий, но keyword-словарь 62.01
   срабатывает первым — нет специфичного слова-якоря.
   Решение: rule-based приоритет для специфичных фраз
   («тепловая энерг» → 35.30, «продуктов питани» → 10.71).

2. 69.10 (Юридические) <-> 70.22 (Консалтинг)
   Причина: слово «консультац» есть в обоих категориях.
   Baseline выбирает 69.10 — оно стоит раньше в словаре.
   Решение: контекстные правила — «ВЭД» / «бизнес» → 70.22;
   «адвокат» / «правов» / «иск» → 69.10.
""")

THRESHOLD = 0.6
gold["needs_review"] = gold["confidence"] < THRESHOLD
print(f"Порог ручной проверки: confidence < {THRESHOLD}")
print(f"На ручную проверку: {gold['needs_review'].sum()} из {len(gold)} ({gold['needs_review'].mean()*100:.1f}%)")
low = gold[gold["needs_review"]][["description","true_code","final_pred","confidence"]]
if len(low):
    display(low)
else:
    print("Все предсказания выше порога — ручная проверка не нужна.")

### 4.7 Сохранение модели

In [ ]:
pickle.dump(vec, open(SRC_DIR / "vectorizer.pkl", "wb"))
pickle.dump(clf, open(SRC_DIR / "model.pkl",      "wb"))
gold[["description","true_code","final_pred","confidence","correct"]].to_csv(
    DATA_DIR / "gold_set_results.csv", index=False)

print("Сохранено:")
print(f"  src/vectorizer.pkl")
print(f"  src/model.pkl")
print(f"  data/gold_set_results.csv")
print()
print("=== ИТОГ ===")
print(f"Keyword Baseline accuracy : {(gold['base_pred']==gold['true_code']).mean():.4f}")
print(f"ML + Rules accuracy       : {gold['correct'].mean():.4f} ({gold['correct'].sum()}/200)")